# **T4.5.2 Geotagging of texts** [WIP] 

* This workflow is part of the [ATRIUM](https://atrium-research.eu/) project.

* This notebook supports only `.txt` file formats, for other formats please check the available notebooks [here](https://github.com/atrium-research/T4.5.2_Geotagging_of_texts/tree/main/notebooks).

* In this example, we will use a preprocessed `Pausanias.txt` file, downloaded from the [Digital Periegesis](https://www.periegesis.org/en/reports.php?projectid=1).

* In this example we will be using the [llama.cpp](https://github.com/ggml-org/llama.cpp) framework.

#### Requirements

In [ ]:
# If you plan to use ToposText as a gazetter then uncomment the 3 lines below:
#!wget https://github.com/atrium-research/T4.5.2_Geotagging_of_texts/releases/download/v.1/topostext.index -P ./data/topostext
#!wget https://github.com/atrium-research/T4.5.2_Geotagging_of_texts/releases/download/v.1/topostext_meta.pkl -P ./data/topostext
#!wget https://github.com/atrium-research/T4.5.2_Geotagging_of_texts/releases/download/v.1/ToposText_gazetteer.json -P ./data/topostext

In [ ]:
# Restart the kernel after the first installation
%pip install -r "https://raw.githubusercontent.com/atrium-research/T4.5.2_Geotagging_of_texts/refs/heads/main/notebooks/requirements.txt"

In [ ]:
import os

# Create the folders
# Move your .txt file inside the 'data' folder
os.makedirs("outputs", exist_ok=True)
os.makedirs("data", exist_ok=True)

### **Step 1: Perform Name Entity Recognition (NER) using our pre-trained model**

In this dataset, each paragraph is denoted by 2 newlines `\n\n`.
Change the next cell structure according to your own dataset.

In [ ]:
import spacy
from tqdm import tqdm
import srsly
import os

# Setups NER location path for pre-trained model
ner_location_model_path = "en_deberta_v3_base_ner_historical_location"

# Setup input and output paths
with open("data/pausanias.txt", "r", encoding="utf-8") as f:
    input_data = [{"text":i.replace("\n","").strip()} for i in f.readlines() if i != "\n"]

ner_output_path = "./outputs/ner_data.jsonl"

In [ ]:
def NER(model_path, in_data, entity):
    ner_model = spacy.load(model_path)
    annotated_data = []
    for row in tqdm(in_data, desc=f"NER for {entity}"):
        sent_nlp = ner_model(row["text"])
        ner_spans = [{"start": span.start_char, "end": span.end_char, "label": entity} for span in sent_nlp.ents]
        if "spans" in row:
            row["spans"] += ner_spans
        else:
            row["spans"] = ner_spans

        annotated_data.append(row)

    return annotated_data

In [ ]:
ner_data = NER(ner_location_model_path, input_data, "LOCATION")

srsly.write_jsonl(ner_output_path, ner_data)

### **Step 2: Recontext the NER predictions using a Large Language Model (LLM)**

In [ ]:
# Run this on your termimal
#!llama-server -hf unsloth/Qwen3.5-9B-GGUF:Q8_0 --reasoning off'

In [ ]:
from openai import OpenAI
import srsly
from tqdm import tqdm

# Setup input and output paths
input_data = list(srsly.read_jsonl("outputs/ner_data.jsonl"))
llm_output_path = "./outputs/llm_data.jsonl"

# Setup OpenAI endpoint
client = OpenAI(
    base_url="http://localhost:8080/v1",
    api_key="1234"
)

In [ ]:
# This is the instructions the LLM will use, you can change it according to your task

system_prompt = """
You are an expert historian and archaeologist.

You will receive:
- Mention: a referenced entity (building, temple, city, region, island, country, monument, harbor, person, or object)
- Context: a short text snippet used only for disambiguation

Task:
Identify the most likely real-world entity referred to by the Mention using the Context, and produce a clean, retrieval-optimized entity description.

Important:
- Use the Context ONLY for disambiguation.
- Do NOT describe or refer to the Context or source text.
- Do NOT explain reasoning or how the entity appears in the text.
- Do NOT write narrative or historical essays.

Output format:
One single sentence describing the entity.

Style rules:
- Be factual and compact
- Include: entity type + location + key identifying feature
- Avoid extra historical narrative details unless essential for identification
- Keep it consistent across all outputs

Output examples:

Kantharos: Kantharos is the main ancient harbor of Piraeus in Athens, Greece.

Athens: Athens is a major city in Greece and the historical center of ancient Greek civilization.

Pompeion: The Pompeion is a public building in ancient Athens used for organizing religious processions.
"""

In [ ]:
def recontext(system_prompt, mention, text, client):
    user_prompt = f'Mention: "{mention}" Context: {text}'
    messages = [{"role": "system", "content": system_prompt},{"role": "user", "content": user_prompt}]
    
    response = client.chat.completions.create(
        model = "unsloth/Qwen3.5-9B-GGUF:Q4_K_M",
        messages=messages,
        max_tokens=81920,
        temperature=1.0,
        top_p=0.95,
        presence_penalty=1.5,
        extra_body={
            "repetition-penalty":1.0,
            "top-k":20,
            "min-p":0.0
        }
    )

    return response.choices[0].message.content

In [ ]:
for row in tqdm(input_data[:30], desc="Text generation"):
    for element in row["spans"]:
        text = row["text"]
        mention = text[element["start"]:element["end"]]
        llm_text = recontext(system_prompt, mention, text, client)
        element.update({"recontext":llm_text})
        
    srsly.write_jsonl(llm_output_path, [row], append=True, append_new_line=False)

### **Step 3: Indexing & fast approximate retrieval**

* The FAISS index was built from [ToposText](https://topostext.org/) database and can be found [here](https://github.com/atrium-research/T4.5.2_Geotagging_of_texts/releases/tag/v.1) along with the metadata file.

In [ ]:
import srsly
import faiss
import pickle
from tqdm import tqdm
from openai import OpenAI
import faiss
import numpy as np

In [ ]:
# Setup input and output paths
input_data = list(srsly.read_jsonl("outputs/llm_data.jsonl"))
llm_output_path = "./outputs/retrieval_data.jsonl"

# Setup OpenAI endpoint
client = OpenAI(
    base_url="http://localhost:8080/v1",
    api_key="1234"
)

# Import ToposText gazetteer
gazetteer = srsly.read_json("data/topostext/ToposText_gazetteer.json")
index = faiss.read_index("data/topostext/topostext.index")
with open("data/topostext/topostext_meta.pkl", "rb") as f:
    metadata = pickle.load(f)

In [ ]:
for row in tqdm(input_data):
    for mention in row.get("spans"):
        
        array = []
        query_text = f'{mention.get("name")}: {mention.get("recontext")}'
        embedding = client.embeddings.create(
            model="Qwen/Qwen3-Embedding-8B-GGUF:Q8_0",
            input=query_text,
            encoding_format="float"
            )
        
        query_vec = np.array(embedding.data[0].embedding).reshape(1, -1)
        
        # Comment this if you want to get the top 1 result without running the 3.1 step
        #distances, indices = index.search(query_vec, 100)

        # Uncomment this if you want to the the top 1 result without running the 3.1 step
        distances, indices = index.search(query_vec, 5)

        result_ids = metadata.get("ids")[indices]

        for id, distance in zip(result_ids[0], distances[0]):
            array.append([gazetteer["features"][list(id)[0]].get("@id").split("/")[-1], str(distance)])

        mention.update({"index_results":array})
    srsly.write_jsonl(llm_output_path, [row], append=True, append_new_line=False)

### **Step 4: Run a Reranker for better results**

* In this examples, we are using the [Qwen3-Reranker-8B](https://huggingface.co/Qwen/Qwen3-Reranker-8B).
* However, you can use either the [Qwen3-Reranker-4B](https://huggingface.co/Qwen/Qwen3-Reranker-4B) or the [Qwen3-Reranker-0.6B](https://huggingface.co/Qwen/Qwen3-Reranker-0.6B) if you have limited resources or any reranker model you want.

In [ ]:
import requests
import srsly
from tqdm import tqdm
from openai import OpenAI
from rapidfuzz import process, fuzz

In [ ]:
# Setup input and output paths
input_data = list(srsly.read_jsonl("./outputs/retrieval_data.jsonl"))
rerank_output_path = "./outputs/rerank_data.jsonl"

data = srsly.read_json("data/topostext/ToposText_gazetteer.json")
data = data["features"]

system_prompt = """
You are given a query and a list of candidate entities. 
Each candidate represents a possible meaning of the query.

Your task is to select the candidate that best matches the query based on its context and description.

Instructions:
- Use the full query, including any descriptive text, to understand the intended meaning.
- Compare the meaning of the query with each candidate.
- Select the single candidate that is the best semantic match.
- Return only the exact candidate string, with no explanation.
"""

client = OpenAI(
    base_url="http://localhost:8080/v1",
    api_key="1234"
)

In [ ]:
def llm_reranker(system_prompt, documents, ids, query):
    messages = [{"role": "system", "content": system_prompt},{"role": "user", "content": f"Query:{query}\nDocuments:{documents}"}]
    
    response = client.chat.completions.create(
        model = "unsloth/gpt-oss-20b-GGUF:F16",
        messages=messages, 
        max_tokens=81920, 
        temperature=1.0,
        top_p=0.95,
        presence_penalty=1.5,
        extra_body={
            "repetition-penalty":1.0,
            "top-k":64,
            "min-p":0.0
        }
    )

    choice = process.extractOne(response.choices[0].message.content, documents, scorer=fuzz.WRatio)[0]
    
    index = documents.index(choice)
    return ids[index]

In [ ]:
for i in tqdm(input_data):
    for mention in i["spans"]:
        topos_text_score = [i[1] for i in mention.get("index_results")]
        if float(topos_text_score[0]) - float(topos_text_score[1]) <= 0.05:
            hash_map = []
            topos_text_ids = [i[0] for i in mention.get("index_results")]

            for id in topos_text_ids:
                for d in data:
                    if d.get("@id").split("/")[-1] == id:
                        title = d.get("properties").get("title")
                        description = d.get("properties").get("description")
                        hash_map.append({id:f"{title}: {description}"})
                        break
                        
            query = mention.get('recontext')
            documents = [list(element.values())[0] for element in hash_map]
            ids = [list(element.keys())[0] for element in hash_map]

            result = llm_reranker(system_prompt, documents, ids, query)
            print(f'query:{query}, index:{mention.get("index_results")[0][0]}, result:{result}')

            mention.update({"reranker_results":result})
        else:
            mention.update({"reranker_results":mention.get("index_results")[0][0]})

    srsly.write_jsonl("outputs/reranker_data.jsonl", [i], append=True, append_new_line=False)

### **Step 5: Create the input for the annotation enviroment**
* In this step, we will create the structured input data format for the annotation enviroment.
* In this example, we choose as our annotation enviroment the [Recogito Studio](https://recogitostudio.org/).
* For the standarized input format, we choose `XML/TEI`.
* However, you can also choose a different input format based on your goals and annotation enviroment.

In [ ]:
#WIP

In [ ]:
# We assume that we executed the 4.1 optional step, comment this if you didn't execute it
data = list(srsly.read_jsonl("files/4_1_pausanias_rerank.jsonl"))

# Uncomment this if you didn't execute the 4.1 optional step
#data = list(srsly.read_jsonl("files/4_pausanias_faiss.jsonl"))

In [ ]:
NS_TEI = "http://www.tei-c.org/ns/1.0"
NS_XML = "http://www.w3.org/XML/1998/namespace"
NSMAP = {None: NS_TEI}

In [ ]:
counter = 1
uid_counter = 0
chapter = 0
current_book = None
current_chapter = None

In [ ]:
# If you want each chapter-book pair to be a different XML run this cell
for i in data:
    book = i.get("book")
    chapter = i.get("chapter")

    if current_chapter is not None and chapter != current_chapter:
        tree = etree.ElementTree(tei)
        tree.write(
            f"books_chapters/pausanias_book_{current_book}_chapter_{current_chapter}.xml",
            xml_declaration=True,
            encoding="utf-8",
            pretty_print=True
        )

        tei = etree.Element("TEI", nsmap=NSMAP, version="3.3.0")
        standoff = etree.SubElement(tei, "standOff", type="recogito_studio_annotations")
        listannotation = etree.SubElement(standoff, "listAnnotation")
        text = etree.SubElement(tei, "text")
        body = etree.SubElement(text, "body")

        counter = 1

        head = etree.SubElement(body, "head")
        head.text = f"Book {book}, Chapter {chapter}"

    if current_chapter is None:
        tei = etree.Element("TEI", nsmap=NSMAP, version="3.3.0")
        standoff = etree.SubElement(tei, "standOff", type="recogito_studio_annotations")
        listannotation = etree.SubElement(standoff, "listAnnotation")
        text = etree.SubElement(tei, "text")
        body = etree.SubElement(text, "body")

        counter = 1

        head = etree.SubElement(body, "head")
        head.text = f"Book {book}, Chapter {chapter}"

    current_book = book
    current_chapter = chapter

    p = etree.SubElement(body, "p")
    p.text = i.get("text")

    for mention in i.get("mentions_tagged"):
        annotation = etree.SubElement(listannotation, "annotation", target=f"/TEI[1]/text[1]/body[1]/p[{str(counter)}]::{str(mention.get("start"))} /TEI[1]/text[1]/body[1]/p[{str(counter)}]::{str(mention.get("end"))}")
        annotation.set(f"{{{NS_XML}}}id", f"UID-FAKE-{uid_counter}")

        # Comment this if you didn't run the optional step
        topos_id = list(mention.get("reranker")[0].keys())[0]
        
        # Uncomment this if you didn't run the optional step
        #topos_id = mention.get("vector_db")[0][0]
        
        rs = etree.SubElement(annotation, "rs", ana=f"https://topostext.org/place/{topos_id}")
        uid_counter += 1

    counter += 1

tree = etree.ElementTree(tei)
tree.write(
    f"books_chapters/pausanias_book_{current_book}_chapter_{current_chapter}.xml",
    xml_declaration=True,
    encoding="utf-8",
    pretty_print=True
)

In [ ]:
# If you want each book to be a different XML run this
for i in data:
    if current_book is not None and i.get("book") != current_book:
        tree = etree.ElementTree(tei)
        tree.write(
            f"books/pausanias_book_{current_book}.xml",
            xml_declaration=True,
            encoding="utf-8",
            pretty_print=True
        )

        tei = etree.Element("TEI", nsmap=NSMAP, version="3.3.0")
        standoff = etree.SubElement(tei, "standOff", type="recogito_studio_annotations")
        listannotation = etree.SubElement(standoff, "listAnnotation")
        text = etree.SubElement(tei, "text")
        body = etree.SubElement(text, "body")

        counter = 1
        chapter = 0 

        head = etree.SubElement(body, "head")
        head.text = f"Book {i.get("book")}"

    current_book = i.get("book")

    if i.get("chapter") != chapter:
        head = etree.SubElement(body, "head")
        head.text = f"Chapter {i.get("chapter")}"
        chapter = i.get("chapter")

    p = etree.SubElement(body, "p")
    p.text = i.get("text")
    for mention in i.get("mentions_tagged"):
        annotation = etree.SubElement(listannotation, "annotation", target=f"/TEI[1]/text[1]/body[1]/p[{str(counter)}]::{str(mention.get("start"))} /TEI[1]/text[1]/body[1]/p[{str(counter)}]::{str(mention.get("end"))}")
        annotation.set(f"{{{NS_XML}}}id", f"UID-FAKE-{uid_counter}")

        # Comment this if you didn't run the optional step
        topos_id = list(mention.get("reranker")[0].keys())[0]
        
        # Uncomment this if you didn't run the optional step
        #topos_id = mention.get("vector_db")[0][0]

        rs = etree.SubElement(annotation, "rs", ana=f"https://topostext.org/place/{topos_id}")
        uid_counter += 1

    counter += 1

tree = etree.ElementTree(tei)
tree.write(
    f"books/pausanias_book_{current_book}.xml",
    xml_declaration=True,
    encoding="utf-8",
    pretty_print=True
)